In [10]:
!pip install google-generativeai pandas

  Using cached google_generativeai-0.8.6-py3-none-any.whl (155 kB)


In [11]:
!pip install -U google-generativeai

In [5]:
import pandas as pd


thread_df = pd.read_csv("../../02_Data/processed/comments_merged_thread.csv")


In [6]:
thread_df

,Unnamed: 0,thread_id,projectID,group,comment_ids,n_comments,merged_comment
0,0,1691589,5787,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...
1,1,1725920,5787,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...
2,2,1725876,5787,0,[1725876],1,about an hour left to go
3,3,1725671,5787,0,[1725671],1,Man 2 hours to go worth staying up to midnight...
4,4,1725512,5787,0,[1725512],1,5 hours to go! Hold on to your butts!
...,...,...,...,...,...,...,...
73029,73029,133497,1143,1,"[133497, 133510, 134222]",3,"Hello. Will it be in Spanish? and if not, woul..."
73030,73030,133314,1143,1,"[133314, 133507]",2,Not sure if people get notifications for repli...
73031,73031,133207,1143,1,"[133207, 133219]",2,The project looks interesting! Who is the desi...
73032,73032,133159,1143,1,"[133159, 133202, 133500]",3,"Hi, is there the future possibility of a solo ..."


In [8]:
thread_df["merged_comment"]

0        WELCOME TO EDEN!  Please check the FAQ and the...
1        @BlackSiteStudios Could a person use larger di...
2                                 about an hour left to go
3        Man 2 hours to go worth staying up to midnight...
4                   5 hours to go!  Hold on to your butts!
                               ...                        
73029    Hello. Will it be in Spanish? and if not, woul...
73030    Not sure if people get notifications for repli...
73031    The project looks interesting! Who is the desi...
73032    Hi, is there the future possibility of a solo ...
73033    Question about the standard vs. innkeepers edi...
Name: merged_comment, Length: 73034, dtype: object

In [33]:
test_df = thread_df.sample(n=30, random_state=42).copy() 

In [12]:
import time
import pandas as pd
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. API 설정
try:
    with open("api_key.txt", "r") as file:
        api_key = file.read().strip() 
    genai.configure(api_key=api_key)
except FileNotFoundError:
    print("api_key.txt 파일 오류.")
    exit()

model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 2. 데이터 준비
target_df = thread_df.copy().reset_index(drop=True)

system_prompt = """You are an expert community manager for a global crowdfunding platform.
Your task is to analyze user comments and classify the core intent into exactly ONE of the following categories:

[Categories]
1. Shipping_Fulfillment
2. Product_Question
3. Praise_Support
4. Complaint_Refund
5. Suggestion_Idea
6. Spam_Irrelevant

[Rules]
- Read the entire conversation thread and select the ONE category that best fits the CORE intent.
- Lines starting with ' ' are replies to the main comment.
- DO NOT provide any explanation.
- ONLY output the exact name of the category."""

# 3. 분류 함수 (병렬용)
def classify_row(args):
    seq_idx, row = args
    try:
        full_prompt = f"{system_prompt}\n\n[Thread Text]\n{row['merged_comment']}"
        response = model.generate_content(full_prompt)
        return seq_idx, response.text.strip()
    except Exception:
        return seq_idx, "Error"

# 4. 병렬 실행 및 진행률 표시
print(f"🚀 총 {len(target_df)}개 데이터 병렬 분류 시작...")

results_dict = {}

# 원하는 최종 컬럼 순서 지정
selected_columns = ['thread_id', 'projectID', 'group', 'n_comments', 'merged_comment', 'category']

# 50개의 스레드를 동시에 사용하여 구글 서버에 요청
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(classify_row, (seq_idx, row)) for seq_idx, (_, row) in enumerate(target_df.iterrows())]
    
    for future in tqdm(futures, total=len(target_df), desc="분류 진행중"):
        seq_idx, category = future.result()
        results_dict[seq_idx] = category

# 5. 최종 저장 단계
target_df['category'] = [results_dict[i] for i in range(len(target_df))]
final_df = target_df[selected_columns]

# 최종 결과 저장
final_df.to_csv("../../02_Data/processed/final_all_results.csv", index=False, encoding="utf-8-sig")
print("\n 모든 분류가 완료되었습니다.")
print(" 최종 저장 컬럼: ['thread_id', 'projectID', 'group', 'n_comments', 'merged_comment', 'category']")

c:\Users\seon\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
c:\Users\seon\anaconda3\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Users\seon\anaconda3\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort bas

🚀 총 73034개 데이터 병렬 분류 시작...


분류 진행중: 100%|██████████| 73034/73034 [1:04:30<00:00, 18.87it/s]



 모든 분류가 완료되었습니다.
 최종 저장 컬럼: ['thread_id', 'projectID', 'group', 'n_comments', 'merged_comment', 'category']


In [ ]:
#오류나면 실행
import pandas as pd
import time
from tqdm import tqdm

# 1. 기존 결과 파일 불러오기
df_result = pd.read_csv("../../02_Data/processed/final_all_results.csv")

# 2. 'Error'로 표시된 행만 추출
error_df = df_result[df_result['category'] == 'Error'].copy()

print(f"❌ 총 {len(error_df)}개의 에러 데이터를 발견했습니다. 재시도 시작...")

# 3. 에러 데이터만 다시 분류하는 루프
for index, row in tqdm(error_df.iterrows(), total=len(error_df), desc="재분류 중"):
    try:
        # 동일한 프롬프트로 재시도
        full_prompt = f"{system_prompt}\n\n[Thread Text]\n{row['thread_text']}"
        response = model.generate_content(full_prompt)
        
        # 성공하면 해당 인덱스의 값을 덮어씌움
        df_result.at[index, 'category'] = response.text.strip()
        time.sleep(0.1) # 서버 과부하 방지
    except Exception:
        # 여전히 실패하면 'Final_Error'로 기록
        df_result.at[index, 'category'] = 'Final_Error'

# 4. 최종 결과 저장
df_result.to_csv("../../02_Data/processed/final_all_results_fixed.csv", index=False, encoding="utf-8-sig")
print(f"\n🎉 재분류 완료! 'final_all_results_fixed.csv' 파일을 확인하세요.")

❌ 총 0개의 에러 데이터를 발견했습니다. 재시도 시작...


재분류 중: 0it [00:00, ?it/s]



🎉 재분류 완료! 'final_all_results_fixed.csv' 파일을 확인하세요.


In [57]:
target_df.head(10)

,thread_id,projectID,thread_text,comment_count,category
0,1050.0,222,and set up a separate forum to complain about ...,3,Suggestion_Idea
1,1394.0,222,Division sounds much larger than section. I t...,1,Suggestion_Idea
2,2395.0,222,Yes please!|Yes please!|Yes!!!,3,Praise_Support
3,2527.0,222,Also REALLY not a fan of ship die and randomiz...,1,Suggestion_Idea
4,3580.0,222,They almost always offer paladin sleeves as an...,1,Suggestion_Idea
5,3605.0,222,"@OutOfTime If you haven't already, you should ...",1,Suggestion_Idea
6,3888.0,222,The other issue was how fragile the dice were ...,1,Complaint_Refund
7,4093.0,222,#Canadianshipping https://boardgamegeek.com/th...,1,Shipping_Fulfillment
8,4096.0,222,#Canadianshipping https://boardgamegeek.com/th...,1,Shipping_Fulfillment
9,4217.0,222,#Canadianshipping https://boardgamegeek.com/th...,1,Shipping_Fulfillment


In [60]:
target_df['category'].value_counts()

Product_Question                                                                                                                                                                                                     34100
Suggestion_Idea                                                                                                                                                                                                      21652
Praise_Support                                                                                                                                                                                                        9169
Shipping_Fulfillment                                                                                                                                                                                                  5473
Complaint_Refund                                                                                                            

In [61]:
# 우리가 설정한 정상 카테고리 목록
valid_categories = [
    'Shipping_Fulfillment', 'Product_Question', 'Praise_Support', 
    'Complaint_Refund', 'Suggestion_Idea', 'Spam_Irrelevant'
]

# 'Error'이거나, 정상 카테고리에 포함되지 않는 모든 행을 추출
error_rows = df_result[~df_result['category'].isin(valid_categories)]

# 이상한 결과만 모아서 보여주기
print(f"🚨 총 {len(error_rows)}개의 이상 행을 발견했습니다.")
print(error_rows[['category', 'thread_text']].head(20))

# 따로 파일로 저장해서 엑셀로 편하게 확인하기
error_rows.to_csv("이상데이터_확인용.csv", index=False, encoding="utf-8-sig")
print("\n💾 '이상데이터_확인용.csv' 파일로 저장했습니다. 엑셀로 열어보세요!")

🚨 총 67개의 이상 행을 발견했습니다.
                                                category  \
1200   Please provide the text of the thread you woul...   
5943   Please provide the thread text you would like ...   
5950   Please provide the thread text you would like ...   
10345  Please provide the thread text you would like ...   
13200  Please provide the text you would like me to a...   
13674  Please provide the thread text you would like ...   
13709  Please provide the thread text you would like ...   
13755  Please provide the thread text you would like ...   
13886  Please provide the thread text so I can classi...   
24153  Please provide the text of the thread you woul...   
32389  Please provide the text you would like me to a...   
32778  Please provide the thread text you would like ...   
32852  Please provide the thread text you would like ...   
33089  Please provide the thread text you would like ...   
33407  Please provide the thread text you would like ...   
33451  Please pro

In [71]:
set(error_rows['projectID'])

{222,
 887,
 889,
 1032,
 2225,
 2460,
 2518,
 2757,
 3493,
 3848,
 4400,
 4475,
 4664,
 4874,
 5030,
 6626,
 6631,
 8203,
 8526}

In [83]:
len(df.loc[df['projectID']==8526])

864

In [ ]:
thread_df.loc[(df['projectID']==8526) & (df['comment_id']=='2595354')]

,projectID,comment_id,parent_id,depth_level,author_id,author_name,creator_id,is_pledge_master,is_backer,backer_number,...,is_pathfinder,has_children,children_count,text,created_at,likes,phaseLabel,campaignStart,campaignEnd,thread_id


In [103]:
thread_df.loc[(thread_df['projectID']==8526)& (thread_df['thread_id']=='2595354')]

,thread_id,projectID,thread_text,comment_count


In [96]:
df.columns

Index(['projectID', 'comment_id', 'parent_id', 'depth_level', 'author_id',
       'author_name', 'creator_id', 'is_pledge_master', 'is_backer',
       'backer_number', 'is_prior_backer', 'is_pathfinder', 'has_children',
       'children_count', 'text', 'created_at', 'likes', 'phaseLabel',
       'campaignStart', 'campaignEnd', 'thread_id'],
      dtype='object')